# Visualize Spatial Calibration

Load a calibration `.npz` from Step 3 and visualize the per-pixel spatial response
overlaid on a reference RGB image. Each SPAD pixel is assigned a unique color; the
alpha at each scan position encodes the response strength.

In [ ]:
from pathlib import Path

import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm, to_rgba
from matplotlib.backends.backend_agg import FigureCanvasAgg as FigureCanvas
import matplotlib.lines as mlines

# ── Configuration ──────────────────────────────────────────────────────────
CALIBRATION_NPZ = Path("path/to/spad_calib_3x3_longrange_bins15_35.npz")
RGB_DIR = Path("path/to/processed_largepatch/aligned_rgb")  # for background image
SHOW_ID = "0488"  # capture ID to use as the background RGB

BG_ALPHA = 1.0
MAX_ALPHA = 0.90
ALPH_LOW_THRESH = 0.025
DOT_SIZE = 22
BG_DARKEN = 0.7
# ──────────────────────────────────────────────────────────────────────────

In [ ]:
# Load calibration
cal = np.load(CALIBRATION_NPZ)
responses = cal["responses"]      # (Sy, Sx, K)
xs = cal["xs"].astype(np.float32) # (K,)
ys = cal["ys"].astype(np.float32) # (K,)

Sy, Sx, N = responses.shape
P = Sy * Sx
print(f"Grid: {Sy}x{Sx} = {P} pixels, {N} scan positions")

# Load background RGB
bg_path = RGB_DIR / f"{SHOW_ID}.png"
bg_img = np.array(Image.open(bg_path).convert("RGB")).astype(np.float32) / 255.0
H, W = bg_img.shape[:2]
print(f"Background: {W}x{H} from {bg_path.name}")

In [ ]:
# Global normalization
vals_all = responses.astype(np.float32)
finite = vals_all[np.isfinite(vals_all)]
vmin = float(np.percentile(finite, 1))
vmax = float(np.percentile(finite, 99.5))

def normalize(vals):
    out = (vals.astype(np.float32) - vmin) / max(vmax - vmin, 1e-8)
    out[~np.isfinite(out)] = 0.0
    return np.clip(out, 0.0, 1.0)

# Per-pixel colors (HSV colormap)
cmap = plt.get_cmap("hsv")
colors = [cmap(i / max(1, P)) for i in range(P)]


def render_layer(xs, ys, rgba, H, W, dpi=150):
    """Render a scatter layer as an RGBA float32 image."""
    fig = plt.Figure(figsize=(W / dpi, H / dpi), dpi=dpi)
    canvas = FigureCanvas(fig)
    ax = fig.add_axes([0, 0, 1, 1])
    ax.set_xlim(0, W)
    ax.set_ylim(H, 0)
    ax.set_axis_off()
    ax.set_facecolor((0, 0, 0, 0))
    fig.patch.set_alpha(0.0)
    ax.scatter(xs, ys, s=DOT_SIZE, c=rgba, linewidths=0)
    canvas.draw()
    return np.asarray(canvas.buffer_rgba(), dtype=np.float32) / 255.0

In [ ]:
# Composite all pixel layers
C_sum = np.zeros((H, W, 3), dtype=np.float32)
A_sum = np.zeros((H, W), dtype=np.float32)

k = 0
for py in range(Sy):
    for col in range(Sx):
        px = (Sx - 1) - col  # horizontal flip
        vals = vals_all[py, px, :]
        u = normalize(vals)

        rgba = np.tile(np.array(to_rgba(colors[k]), dtype=np.float32), (N, 1))
        alpha_vals = (u * MAX_ALPHA).astype(np.float32)
        alpha_vals[alpha_vals < ALPH_LOW_THRESH] = 0.0
        rgba[:, 3] = alpha_vals

        layer = render_layer(xs, ys, rgba, H, W)
        a = layer[..., 3]
        C_sum += layer[..., :3] * a[..., None]
        A_sum += a
        k += 1
        print(f"  Rendered pixel {k}/{P}", end="\r")

C_out = C_sum / np.maximum(A_sum[..., None], 1e-8)
A_out = np.clip(1.0 - np.exp(-A_sum), 0, 1)
print(f"\nDone. Composited {P} pixel layers.")

In [ ]:
# Final overlay on background RGB
bg_disp = np.clip(bg_img * BG_DARKEN, 0, 1) * BG_ALPHA + (1 - BG_ALPHA)
out = C_out * A_out[..., None] + bg_disp * (1 - A_out[..., None])
out = np.clip(out, 0, 1)

# Plot with legend
fig, ax = plt.subplots(1, 1, figsize=(14, 8))
ax.imshow(out)
ax.set_axis_off()

legend_handles = [
    mlines.Line2D([], [], color=colors[i], marker="o", linestyle="None",
                  markersize=10, label=f"Pixel {i + 1}")
    for i in range(P)
]
ax.legend(handles=legend_handles, loc="upper right", fontsize=9,
          framealpha=0.8, title="SPAD Pixels")

ax.set_title(f"Spatial Calibration — {Sy}x{Sx} SPAD ({N} scan positions)")
plt.tight_layout()
plt.show()